# DQ-02 · orders.csv
Checks: null rate, duplicates, FK → customers / geography, domain values, temporal.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── helpers ───────────────────────────────────────────────────────────────────
issues = []

def flag(label, mask_or_count, df=None, show_sample=True):
    count = int(mask_or_count.sum()) if hasattr(mask_or_count, 'sum') else int(mask_or_count)
    total = len(df) if df is not None else None
    pct   = f' ({count/total*100:.2f}%)' if total else ''
    status = '❌ ISSUE' if count > 0 else '✅ OK'
    print(f'{status}  {label}: {count:,}{pct}')
    if count > 0:
        issues.append(label)
        if show_sample and df is not None and hasattr(mask_or_count, 'sum'):
            sample = df[mask_or_count].head(3)
            print(sample.to_string(index=False))
    return count

def summary():
    print()
    if issues:
        print(f'══ {len(issues)} issue(s) found ══')
        for i in issues: print(f'  • {i}')
    else:
        print('══ All checks passed ══')


In [ ]:
orders = pd.read_csv('orders.csv', parse_dates=['order_date'])
cust   = pd.read_csv('customers.csv', parse_dates=['signup_date'])
geo    = pd.read_csv('geography.csv')
print(f'Shape: {orders.shape}')
orders.head(3)

## 1. Null rate

In [ ]:
null_counts = orders.isnull().sum()
print(pd.DataFrame({'null_count': null_counts, 'null_%': (null_counts/len(orders)*100).round(2)}))

## 2. Duplicate order_id

In [ ]:
flag('Duplicate order_id', orders.duplicated(subset='order_id'), orders)

## 3. FK: customer_id → customers

In [ ]:
valid_cust = set(cust['customer_id'])
flag('customer_id not in customers', ~orders['customer_id'].isin(valid_cust), orders)

## 4. FK: zip → geography

In [ ]:
valid_zips = set(geo['zip'])
flag('order zip not in geography', ~orders['zip'].isin(valid_zips), orders)

## 5. Domain values

In [ ]:
VALID_STATUS  = {'delivered','cancelled','returned','shipped','paid','created'}
VALID_PAYMENT = {'credit_card','paypal','cod','apple_pay','bank_transfer'}
VALID_DEVICE  = {'mobile','desktop','tablet'}
VALID_SOURCE  = {'organic_search','paid_search','social_media','email_campaign','referral','direct'}

flag('Invalid order_status',    ~orders['order_status'].isin(VALID_STATUS),  orders)
flag('Invalid payment_method',  ~orders['payment_method'].isin(VALID_PAYMENT), orders)
flag('Invalid device_type',     ~orders['device_type'].isin(VALID_DEVICE),   orders)
flag('Invalid order_source',    ~orders['order_source'].isin(VALID_SOURCE),  orders)

## 6. Temporal: order_date ≥ signup_date

In [ ]:
merged = orders.merge(cust[['customer_id','signup_date']], on='customer_id', how='left')
early  = merged['order_date'] < merged['signup_date']
flag('order_date < customer signup_date', early, merged)

## 7. Temporal: order_date range

In [ ]:
flag('order_date before 2012-01-01', orders['order_date'] < pd.Timestamp('2012-01-01'), orders)
flag('order_date after  2022-12-31', orders['order_date'] > pd.Timestamp('2022-12-31'), orders)

## Summary

In [ ]:
summary()